# 06 — Production Video Identity Pipeline

## Competition configuration

This is the final production pipeline selected from notebooks 03–05.

```text
Every 15 source frames
        ↓
SCRFD evaluates all 15
        ↓
quality ranking using:
- face sharpness
- frontalness from SCRFD landmarks
- SCRFD confidence
- face size
- exposure
- full-frame sharpness
        ↓
┌───────────────────────────────┐
│ Top 2 frames → PersonID       │
│ SCRFD faces → AdaFace IR101   │
│ → temporal tracklets          │
│ → quality-weighted pooling    │
│ → global Person_ID clustering │
└───────────────────────────────┘

Best 1 frame
        ↓
activity + sentiment handoff
```

### Models

- **SCRFD-10G / AntelopeV2** — face detection + 5 facial landmarks
- **AdaFace IR101 WebFace12M** — 512-D face identity embeddings
- AntelopeV2's recognition model is **not** used.

### Team outputs

RJ receives:

```text
Handoffs/sentiment_input_clean.csv
```

with the exact schema:

```text
video_title
video_filename
frame_number
person_id
image_path
source_time_seconds
det_score
```

Marion receives:

```text
Handoffs/activity_frame_person_map.csv
```

with the exact schema:

```text
video_title
video_filename
frame_number
frame_id
person_id
source_time_seconds
full_frame_path
face_crop_path
face_bbox_x1
face_bbox_y1
face_bbox_x2
face_bbox_y2
det_score
```

Marion should run activity inference from:

```text
Handoffs/activity_frames_unique.csv
```

which has **one row per selected frame**, then join results back to `activity_frame_person_map.csv`.

## 0. Configuration

Start in `test` mode. After the test succeeds, switch only `RUN_MODE` to `"all"`.

`RESET_RUN = True` deletes this notebook's current output for the selected mode.

In [ ]:
from pathlib import Path
import os
import re
import sys
import json
import math
import shutil
import random
import hashlib
import warnings
import types

import cv2
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

from IPython.display import display


# ============================================================
# CORE CONFIGURATION
# ============================================================

RUN_MODE = "test"          # "test" or "all"
RESET_RUN = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if RUN_MODE not in {
    "test",
    "all",
}:
    raise ValueError(
        "RUN_MODE must be 'test' or 'all'."
    )


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(".")
GIVEN_DATA = ROOT / "GivenData"
DATA = ROOT / "Data"
MODELS = ROOT / "Models"

RUN_ROOT = (
    DATA /
    "IdentityPipelineTop2" /
    RUN_MODE
)

FRAMES_DIR = (
    RUN_ROOT /
    "Frames"
)

FACE_CROPS_DIR = (
    RUN_ROOT /
    "FaceCrops"
)

ALIGNED_DIR = (
    RUN_ROOT /
    "IdentityAligned"
)

EMBEDDINGS_DIR = (
    RUN_ROOT /
    "Embeddings"
)

RESULTS_DIR = (
    RUN_ROOT /
    "Results"
)

HANDOFF_DIR = (
    RUN_ROOT /
    "Handoffs"
)

REP_FACES_DIR = (
    RUN_ROOT /
    "RepresentativeFaces"
)


if RESET_RUN and RUN_ROOT.exists():
    shutil.rmtree(
        RUN_ROOT
    )


for folder in [
    RUN_ROOT,
    FRAMES_DIR,
    FACE_CROPS_DIR,
    ALIGNED_DIR,
    EMBEDDINGS_DIR,
    RESULTS_DIR,
    HANDOFF_DIR,
    REP_FACES_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


if not GIVEN_DATA.exists():
    raise FileNotFoundError(
        f"Missing GivenData folder: {GIVEN_DATA.resolve()}"
    )


# ============================================================
# TEST MODE
# ============================================================

TEST_VIDEO_COUNT = 4
TEST_CLIPS_PER_VIDEO = 2
TEST_CLIP_SECONDS = 30.0
TEST_VIDEO_NAMES = []


# ============================================================
# 15-FRAME WINDOW
# ============================================================

FRAMES_PER_WINDOW = 15

# Top 2 frames contribute to PersonID.
IDENTITY_TOP_N = 2

# Best 1 frame goes to RJ + Marion.
DOWNSTREAM_TOP_N = 1


# ============================================================
# SCRFD FILTERS
# ============================================================

MIN_DET_SCORE = 0.60
MIN_FACE_SIZE_PIXELS = 50
FACE_CROP_MARGIN = 0.25
SCRFD_DET_SIZE = (
    640,
    640,
)


# ============================================================
# FACE QUALITY WEIGHTS
# ============================================================

FACE_SHARPNESS_WEIGHT = 0.40
FACE_FRONTAL_WEIGHT = 0.25
FACE_DETECTION_WEIGHT = 0.15
FACE_SIZE_WEIGHT = 0.10
FACE_EXPOSURE_WEIGHT = 0.10


# ============================================================
# FRAME QUALITY WEIGHTS
# ============================================================

FRAME_MEDIAN_FACE_WEIGHT = 0.40
FRAME_WORST_FACE_WEIGHT = 0.15
FRAME_FACE_COUNT_WEIGHT = 0.25
FRAME_SHARPNESS_WEIGHT = 0.20


# ============================================================
# IDENTITY / TRACKLETS
# ============================================================

LOCAL_TRACK_SIMILARITY = 0.62
MAX_TRACK_GAP_FRAMES = 14
MAX_CENTER_DISTANCE_NORM = 0.35
SPATIAL_PENALTY = 0.05

TRACKLET_TOP_K = 2
GLOBAL_TRACKLET_SIMILARITY = 0.76

EMBED_BATCH_SIZE = 32


# ============================================================
# OUTPUT QUALITY
# ============================================================

FULL_FRAME_JPEG_QUALITY = 92
FACE_CROP_JPEG_QUALITY = 95
ALIGNED_JPEG_QUALITY = 100

# Save extraction progress regularly.
CHECKPOINT_WINDOWS = 25


print("Project:", ROOT.resolve())
print("Run mode:", RUN_MODE)
print("Output:", RUN_ROOT.resolve())
print("Identity frames/window:", IDENTITY_TOP_N)
print("Downstream frames/window:", DOWNSTREAM_TOP_N)

## 1. Video inventory

Source videos are read directly from:

```text
GivenData/
```

Frame numbers remain the **exact zero-based source-video frame indices**.

In [ ]:
def safe_slug(
    text,
    max_len=55,
):
    text = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(
            text
        ),
    ).strip(
        "_"
    )

    return (
        text[
            :max_len
        ]
        if text
        else "video"
    )


video_paths = sorted(
    GIVEN_DATA.glob(
        "*.mp4"
    )
)

if not video_paths:
    raise FileNotFoundError(
        f"No mp4 files found in {GIVEN_DATA.resolve()}"
    )


inventory_rows = []

for i, video_path in enumerate(
    tqdm(
        video_paths,
        desc="Inspecting videos",
    ),
    start=1,
):

    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )

    if not cap.isOpened():
        print(
            "WARNING: could not open",
            video_path.name,
        )
        continue

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    cap.release()

    duration_seconds = (
        frame_count
        /
        fps
        if fps > 0
        else 0.0
    )

    digest = hashlib.sha1(
        video_path.name.encode(
            "utf-8"
        )
    ).hexdigest()[
        :6
    ]

    video_id = (
        f"V{i:03d}_"
        f"{safe_slug(video_path.stem)}_"
        f"{digest}"
    )

    inventory_rows.append({
        "video_id":
            video_id,

        "source_video":
            video_path.name,

        "source_path":
            str(
                video_path
            ),

        "duration_seconds":
            duration_seconds,

        "duration_minutes":
            duration_seconds
            /
            60.0,

        "reported_fps":
            fps,

        "frame_count":
            frame_count,

        "width":
            width,

        "height":
            height,
    })


df_videos = pd.DataFrame(
    inventory_rows
)

VIDEO_INVENTORY_CSV = (
    RESULTS_DIR /
    "video_inventory.csv"
)

df_videos.to_csv(
    VIDEO_INVENTORY_CSV,
    index=False,
)


display(
    df_videos[
        [
            "video_id",
            "source_video",
            "duration_minutes",
            "reported_fps",
            "frame_count",
            "width",
            "height",
        ]
    ]
)

print()
print(
    "Videos:",
    len(
        df_videos
    ),
)

print(
    "Total source minutes:",
    round(
        float(
            df_videos[
                "duration_minutes"
            ].sum()
        ),
        2,
    ),
)

## 2. Build test/all analysis ranges

Test mode uses the same diverse-video strategy as notebook 03.

All mode processes every source video from beginning to end.

In [ ]:
RANGES_CSV = (
    RESULTS_DIR /
    "analysis_ranges.csv"
)

rng = np.random.default_rng(
    SEED
)

range_rows = []


def choose_test_videos(
    df,
):
    if TEST_VIDEO_NAMES:

        selected = df[
            df[
                "source_video"
            ].isin(
                TEST_VIDEO_NAMES
            )
        ].copy()

        missing = (
            set(
                TEST_VIDEO_NAMES
            )
            -
            set(
                selected[
                    "source_video"
                ]
            )
        )

        if missing:
            raise ValueError(
                f"TEST_VIDEO_NAMES not found: {sorted(missing)}"
            )

        return selected


    ordered = (
        df.sort_values(
            "duration_seconds",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    count = min(
        TEST_VIDEO_COUNT,
        len(
            ordered
        ),
    )

    if count == len(
        ordered
    ):
        return ordered

    positions = np.unique(
        np.round(
            np.linspace(
                0,
                len(
                    ordered
                )
                -
                1,
                count,
            )
        ).astype(
            int
        )
    )

    return ordered.iloc[
        positions
    ].copy()


if RUN_MODE == "test":

    selected_videos = choose_test_videos(
        df_videos
    )

    counter = 0

    for _, video in (
        selected_videos.iterrows()
    ):

        duration = float(
            video[
                "duration_seconds"
            ]
        )

        if duration <= TEST_CLIP_SECONDS:

            starts = [
                0.0
            ]

        else:

            bins = np.linspace(
                0.0,
                duration,
                TEST_CLIPS_PER_VIDEO
                +
                1,
            )

            starts = []

            for b in range(
                TEST_CLIPS_PER_VIDEO
            ):

                region_start = float(
                    bins[
                        b
                    ]
                )

                region_end = float(
                    bins[
                        b + 1
                    ]
                )

                latest_start = min(
                    region_end
                    -
                    TEST_CLIP_SECONDS,

                    duration
                    -
                    TEST_CLIP_SECONDS,
                )

                latest_start = max(
                    region_start,
                    latest_start,
                )

                if (
                    latest_start
                    <=
                    region_start
                ):

                    start = (
                        region_start
                    )

                else:

                    start = float(
                        rng.uniform(
                            region_start,
                            latest_start,
                        )
                    )

                starts.append(
                    start
                )


        for start in starts:

            counter += 1

            end = min(
                duration,
                start
                +
                TEST_CLIP_SECONDS,
            )

            range_rows.append({
                "range_id":
                    f"R{counter:04d}",

                "video_id":
                    video[
                        "video_id"
                    ],

                "source_video":
                    video[
                        "source_video"
                    ],

                "source_path":
                    video[
                        "source_path"
                    ],

                "range_start_seconds":
                    start,

                "range_end_seconds":
                    end,

                "range_duration_seconds":
                    end
                    -
                    start,
            })


else:

    for counter, (
        _,
        video,
    ) in enumerate(
        df_videos.iterrows(),
        start=1,
    ):

        range_rows.append({
            "range_id":
                f"R{counter:04d}",

            "video_id":
                video[
                    "video_id"
                ],

            "source_video":
                video[
                    "source_video"
                ],

            "source_path":
                video[
                    "source_path"
                ],

            "range_start_seconds":
                0.0,

            "range_end_seconds":
                float(
                    video[
                        "duration_seconds"
                    ]
                ),

            "range_duration_seconds":
                float(
                    video[
                        "duration_seconds"
                    ]
                ),
        })


df_ranges = pd.DataFrame(
    range_rows
)

df_ranges.to_csv(
    RANGES_CSV,
    index=False,
)

display(
    df_ranges
)

print(
    "Analysis ranges:",
    len(
        df_ranges
    ),
)

print(
    "Analysis minutes:",
    round(
        float(
            df_ranges[
                "range_duration_seconds"
            ].sum()
        )
        /
        60.0,
        2,
    ),
)

## 3. Load SCRFD

SCRFD is used on **all 15 frames** for:

- face detection
- 5-point landmarks
- frontalness scoring
- downstream face bounding boxes

It does not assign legal names.

In [ ]:
from insightface.app import FaceAnalysis
from insightface.utils.face_align import norm_crop


INSIGHTFACE_ROOT = (
    MODELS /
    "insightface"
)

SCRFD_FILE = (
    INSIGHTFACE_ROOT /
    "models" /
    "antelopev2" /
    "scrfd_10g_bnkps.onnx"
)


if not SCRFD_FILE.exists():
    raise FileNotFoundError(
        "Missing SCRFD model:\n"
        f"{SCRFD_FILE.resolve()}"
    )


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


face_detector = FaceAnalysis(
    name="antelopev2",
    root=str(
        INSIGHTFACE_ROOT
    ),
    allowed_modules=[
        "detection"
    ],
    providers=[
        "CPUExecutionProvider"
    ],
)


face_detector.prepare(
    ctx_id=0,
    det_size=SCRFD_DET_SIZE,
)


print(
    "SCRFD ready:",
    SCRFD_FILE.resolve(),
)

## 4. Quality functions

The frontal score is an approximate pose-quality score derived from SCRFD's five landmarks.

A higher score means the landmarks are more symmetric around the nose and therefore more useful for frontal identity recognition.

In [ ]:
def normalize_vector(
    vector,
):
    vector = np.asarray(
        vector,
        dtype=np.float32,
    )

    norm = np.linalg.norm(
        vector
    )

    if norm <= 0:
        return vector

    return (
        vector
        /
        norm
    )


def percentile_rank(
    series,
):
    if len(
        series
    ) <= 1:

        return pd.Series(
            np.ones(
                len(
                    series
                ),
                dtype=float,
            ),
            index=series.index,
        )

    return (
        series
        .rank(
            method="average",
            pct=True,
        )
        .astype(
            float
        )
    )


def variance_of_laplacian(
    image,
):
    if (
        image is None
        or
        image.size == 0
    ):
        return 0.0

    if image.ndim == 3:

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY,
        )

    else:

        gray = image

    return float(
        cv2.Laplacian(
            gray,
            cv2.CV_64F,
        ).var()
    )


def exposure_score(
    image,
):
    if (
        image is None
        or
        image.size == 0
    ):
        return 0.0

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY,
    )

    mean_value = float(
        gray.mean()
    )

    dark_fraction = float(
        (
            gray <= 10
        ).mean()
    )

    bright_fraction = float(
        (
            gray >= 245
        ).mean()
    )

    brightness = math.exp(
        -(
            (
                mean_value
                -
                128.0
            )
            /
            85.0
        ) ** 2
    )

    clipping = max(
        0.0,
        1.0
        -
        (
            dark_fraction
            +
            bright_fraction
        ),
    )

    return float(
        np.clip(
            0.60
            *
            brightness
            +
            0.40
            *
            clipping,
            0.0,
            1.0,
        )
    )


def frontal_score_from_landmarks(
    kps,
):
    kps = np.asarray(
        kps,
        dtype=np.float32,
    )

    if kps.shape != (
        5,
        2,
    ):
        return 0.0

    left_eye = kps[
        0
    ]

    right_eye = kps[
        1
    ]

    nose = kps[
        2
    ]

    left_mouth = kps[
        3
    ]

    right_mouth = kps[
        4
    ]

    eps = 1e-6

    left_eye_dist = np.linalg.norm(
        nose
        -
        left_eye
    )

    right_eye_dist = np.linalg.norm(
        nose
        -
        right_eye
    )

    eye_asymmetry = abs(
        left_eye_dist
        -
        right_eye_dist
    ) / (
        left_eye_dist
        +
        right_eye_dist
        +
        eps
    )

    left_mouth_dist = np.linalg.norm(
        nose
        -
        left_mouth
    )

    right_mouth_dist = np.linalg.norm(
        nose
        -
        right_mouth
    )

    mouth_asymmetry = abs(
        left_mouth_dist
        -
        right_mouth_dist
    ) / (
        left_mouth_dist
        +
        right_mouth_dist
        +
        eps
    )

    eye_vector = (
        right_eye
        -
        left_eye
    )

    eye_length_sq = float(
        np.dot(
            eye_vector,
            eye_vector,
        )
    )

    if eye_length_sq <= eps:
        return 0.0

    nose_fraction = float(
        np.dot(
            nose
            -
            left_eye,
            eye_vector,
        )
        /
        eye_length_sq
    )

    eye_center_error = min(
        1.0,
        abs(
            nose_fraction
            -
            0.5
        )
        *
        2.0,
    )

    mouth_mid = (
        left_mouth
        +
        right_mouth
    ) / 2.0

    eye_mid = (
        left_eye
        +
        right_eye
    ) / 2.0

    face_axis = (
        mouth_mid
        -
        eye_mid
    )

    axis_length = float(
        np.linalg.norm(
            face_axis
        )
    )

    eye_distance = float(
        np.linalg.norm(
            right_eye
            -
            left_eye
        )
    )

    if (
        axis_length <= eps
        or
        eye_distance <= eps
    ):
        return 0.0

    axis_unit = (
        face_axis
        /
        axis_length
    )

    perpendicular = np.array(
        [
            -axis_unit[
                1
            ],
            axis_unit[
                0
            ],
        ],
        dtype=np.float32,
    )

    side_offset = abs(
        float(
            np.dot(
                nose
                -
                eye_mid,
                perpendicular,
            )
        )
    ) / (
        eye_distance
        +
        eps
    )

    side_error = min(
        1.0,
        side_offset
        /
        0.35,
    )

    error = (
        0.30
        *
        min(
            1.0,
            eye_asymmetry
            /
            0.30,
        )
        +
        0.25
        *
        min(
            1.0,
            mouth_asymmetry
            /
            0.30,
        )
        +
        0.20
        *
        eye_center_error
        +
        0.25
        *
        side_error
    )

    return float(
        np.clip(
            1.0
            -
            error,
            0.0,
            1.0,
        )
    )


def face_size_score(
    face_width,
    face_height,
    frame_width,
    frame_height,
):
    ratio = (
        min(
            face_width,
            face_height,
        )
        /
        max(
            1.0,
            min(
                frame_width,
                frame_height,
            ),
        )
    )

    return float(
        np.clip(
            ratio
            /
            0.15,
            0.0,
            1.0,
        )
    )


def expand_bbox(
    bbox,
    frame_width,
    frame_height,
    margin,
):
    x1, y1, x2, y2 = [
        float(v)
        for v in bbox
    ]

    w = (
        x2
        -
        x1
    )

    h = (
        y2
        -
        y1
    )

    x1 -= (
        w
        *
        margin
    )

    y1 -= (
        h
        *
        margin
    )

    x2 += (
        w
        *
        margin
    )

    y2 += (
        h
        *
        margin
    )

    x1 = max(
        0,
        int(
            round(
                x1
            )
        ),
    )

    y1 = max(
        0,
        int(
            round(
                y1
            )
        ),
    )

    x2 = min(
        frame_width,
        int(
            round(
                x2
            )
        ),
    )

    y2 = min(
        frame_height,
        int(
            round(
                y2
            )
        ),
    )

    if (
        x2 <= x1
        or
        y2 <= y1
    ):
        return None

    return (
        x1,
        y1,
        x2,
        y2,
    )

## 5. SCRFD scan + Top-2 selection

Every 15-frame window is evaluated with SCRFD.

Only the **Top 2 frames** are saved for identity. Only the **best 1 frame** is retained as the downstream activity/sentiment frame.

This is the expensive video pass, so progress is checkpointed.

In [ ]:
WINDOWS_CSV = (
    RESULTS_DIR /
    "window_selections.csv"
)

IDENTITY_FACES_CSV = (
    RESULTS_DIR /
    "top2_identity_face_observations.csv"
)

PROCESSED_WINDOWS_JSON = (
    RESULTS_DIR /
    "processed_windows.json"
)


if WINDOWS_CSV.exists():

    df_window_results = pd.read_csv(
        WINDOWS_CSV
    )

else:

    df_window_results = pd.DataFrame()


if IDENTITY_FACES_CSV.exists():

    df_identity_faces_raw = pd.read_csv(
        IDENTITY_FACES_CSV
    )

else:

    df_identity_faces_raw = pd.DataFrame()


if PROCESSED_WINDOWS_JSON.exists():

    processed_windows = set(
        json.loads(
            PROCESSED_WINDOWS_JSON.read_text(
                encoding="utf-8"
            )
        )
    )

else:

    processed_windows = set()


def append_checkpoint(
    window_rows,
    face_rows,
):
    global df_window_results
    global df_identity_faces_raw

    if window_rows:

        new_windows = pd.DataFrame(
            window_rows
        )

        if len(
            df_window_results
        ):

            df_window_results = pd.concat(
                [
                    df_window_results,
                    new_windows,
                ],
                ignore_index=True,
            )

        else:

            df_window_results = new_windows

        df_window_results = (
            df_window_results
            .drop_duplicates(
                "window_id",
                keep="last",
            )
            .reset_index(
                drop=True
            )
        )

        df_window_results.to_csv(
            WINDOWS_CSV,
            index=False,
        )


    if face_rows:

        new_faces = pd.DataFrame(
            face_rows
        )

        if len(
            df_identity_faces_raw
        ):

            df_identity_faces_raw = pd.concat(
                [
                    df_identity_faces_raw,
                    new_faces,
                ],
                ignore_index=True,
            )

        else:

            df_identity_faces_raw = new_faces

        df_identity_faces_raw = (
            df_identity_faces_raw
            .drop_duplicates(
                "observation_id",
                keep="last",
            )
            .reset_index(
                drop=True
            )
        )

        df_identity_faces_raw.to_csv(
            IDENTITY_FACES_CSV,
            index=False,
        )


pending_window_rows = []
pending_face_rows = []
windows_since_checkpoint = 0


for _, range_row in tqdm(
    df_ranges.iterrows(),
    total=len(
        df_ranges
    ),
    desc="Video ranges",
):

    range_id = str(
        range_row[
            "range_id"
        ]
    )

    video_id = str(
        range_row[
            "video_id"
        ]
    )

    source_video = str(
        range_row[
            "source_video"
        ]
    )

    source_path = Path(
        str(
            range_row[
                "source_path"
            ]
        )
    )

    cap = cv2.VideoCapture(
        str(
            source_path
        )
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open {source_path}"
        )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if fps <= 0:

        cap.release()

        raise RuntimeError(
            f"Invalid FPS: {source_video}"
        )


    start_frame = max(
        0,
        int(
            round(
                float(
                    range_row[
                        "range_start_seconds"
                    ]
                )
                *
                fps
            )
        ),
    )

    end_frame = min(
        total_frames
        -
        1,

        int(
            round(
                float(
                    range_row[
                        "range_end_seconds"
                    ]
                )
                *
                fps
            )
        ),
    )


    window_number = 0


    for window_start in tqdm(
        range(
            start_frame,
            end_frame
            +
            1,
            FRAMES_PER_WINDOW,
        ),
        desc=source_video[
            :45
        ],
        leave=False,
    ):

        window_number += 1

        window_id = (
            f"{range_id}_"
            f"W{window_number:06d}"
        )

        if window_id in processed_windows:
            continue


        window_end = min(
            window_start
            +
            FRAMES_PER_WINDOW
            -
            1,
            end_frame,
        )


        frame_records = []


        # Seek once at the start of this 15-frame block,
        # then decode sequentially. This is much faster than
        # seeking the compressed MP4 separately for every frame.
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(
                window_start
            ),
        )

        for frame_number in range(
            window_start,
            window_end
            +
            1,
        ):

            ok, frame = cap.read()

            if (
                not ok
                or
                frame is None
            ):
                continue


            frame_height, frame_width = (
                frame.shape[
                    :2
                ]
            )

            frame_id = (
                f"{video_id}_"
                f"F{frame_number:09d}"
            )

            source_time_seconds = (
                frame_number
                /
                fps
            )

            frame_sharpness = (
                variance_of_laplacian(
                    frame
                )
            )

            detections = face_detector.get(
                frame
            )

            faces = []
            face_number = 0


            for face in detections:

                det_score = float(
                    getattr(
                        face,
                        "det_score",
                        0.0,
                    )
                )

                if (
                    det_score
                    <
                    MIN_DET_SCORE
                ):
                    continue


                bbox = np.asarray(
                    face.bbox,
                    dtype=float,
                )

                x1, y1, x2, y2 = [
                    int(
                        round(v)
                    )
                    for v in bbox
                ]

                x1 = max(
                    0,
                    min(
                        x1,
                        frame_width
                        -
                        1,
                    ),
                )

                y1 = max(
                    0,
                    min(
                        y1,
                        frame_height
                        -
                        1,
                    ),
                )

                x2 = max(
                    x1
                    +
                    1,

                    min(
                        x2,
                        frame_width,
                    ),
                )

                y2 = max(
                    y1
                    +
                    1,

                    min(
                        y2,
                        frame_height,
                    ),
                )

                face_width = (
                    x2
                    -
                    x1
                )

                face_height = (
                    y2
                    -
                    y1
                )

                if (
                    min(
                        face_width,
                        face_height,
                    )
                    <
                    MIN_FACE_SIZE_PIXELS
                ):
                    continue


                kps = getattr(
                    face,
                    "kps",
                    None,
                )

                if kps is None:
                    continue

                kps = np.asarray(
                    kps,
                    dtype=np.float32,
                )

                if kps.shape != (
                    5,
                    2,
                ):
                    continue


                crop_bbox = expand_bbox(
                    (
                        x1,
                        y1,
                        x2,
                        y2,
                    ),
                    frame_width,
                    frame_height,
                    FACE_CROP_MARGIN,
                )

                if crop_bbox is None:
                    continue

                cx1, cy1, cx2, cy2 = (
                    crop_bbox
                )

                crop = frame[
                    cy1:cy2,
                    cx1:cx2,
                ]

                if (
                    crop is None
                    or
                    crop.size == 0
                ):
                    continue


                aligned = norm_crop(
                    frame,
                    landmark=kps,
                    image_size=112,
                )

                observation_id = (
                    f"{frame_id}_"
                    f"FACE{face_number:02d}"
                )

                face_number += 1


                faces.append({
                    "observation_id":
                        observation_id,

                    "frame_id":
                        frame_id,

                    "window_id":
                        window_id,

                    "range_id":
                        range_id,

                    "video_id":
                        video_id,

                    "source_video":
                        source_video,

                    "source_path":
                        str(
                            source_path
                        ),

                    "source_frame_index":
                        int(
                            frame_number
                        ),

                    "source_time_seconds":
                        float(
                            source_time_seconds
                        ),

                    "reported_fps":
                        fps,

                    "frame_width":
                        frame_width,

                    "frame_height":
                        frame_height,

                    "face_bbox_x1":
                        int(
                            x1
                        ),

                    "face_bbox_y1":
                        int(
                            y1
                        ),

                    "face_bbox_x2":
                        int(
                            x2
                        ),

                    "face_bbox_y2":
                        int(
                            y2
                        ),

                    "center_x_norm":
                        float(
                            (
                                x1 + x2
                            )
                            /
                            2.0
                            /
                            frame_width
                        ),

                    "center_y_norm":
                        float(
                            (
                                y1 + y2
                            )
                            /
                            2.0
                            /
                            frame_height
                        ),

                    "det_score":
                        det_score,

                    "sharpness_laplacian":
                        variance_of_laplacian(
                            aligned
                        ),

                    "frontal_score":
                        frontal_score_from_landmarks(
                            kps
                        ),

                    "exposure_score":
                        exposure_score(
                            aligned
                        ),

                    "face_size_score":
                        face_size_score(
                            face_width,
                            face_height,
                            frame_width,
                            frame_height,
                        ),

                    "_crop":
                        crop,

                    "_aligned":
                        aligned,
                })


            frame_records.append({
                "frame_id":
                    frame_id,

                "window_id":
                    window_id,

                "source_frame_index":
                    int(
                        frame_number
                    ),

                "source_time_seconds":
                    float(
                        source_time_seconds
                    ),

                "full_frame_sharpness":
                    float(
                        frame_sharpness
                    ),

                "frame":
                    frame,

                "faces":
                    faces,
            })


        if not frame_records:

            processed_windows.add(
                window_id
            )

            continue


        # ----------------------------------------------------
        # FACE SCORES ACROSS THIS WINDOW
        # ----------------------------------------------------

        all_face_records = [
            face
            for frame_record
            in frame_records
            for face
            in frame_record[
                "faces"
            ]
        ]


        if all_face_records:

            temp_faces = pd.DataFrame(
                [
                    {
                        "observation_id":
                            face[
                                "observation_id"
                            ],

                        "sharpness_laplacian":
                            face[
                                "sharpness_laplacian"
                            ],
                    }
                    for face
                    in all_face_records
                ]
            )

            temp_faces[
                "sharpness_score"
            ] = percentile_rank(
                np.log1p(
                    temp_faces[
                        "sharpness_laplacian"
                    ].clip(
                        lower=0
                    )
                )
            )

            sharpness_map = dict(
                zip(
                    temp_faces[
                        "observation_id"
                    ],
                    temp_faces[
                        "sharpness_score"
                    ],
                )
            )

            for face in all_face_records:

                face[
                    "sharpness_score"
                ] = float(
                    sharpness_map[
                        face[
                            "observation_id"
                        ]
                    ]
                )

                face[
                    "face_quality_score"
                ] = float(
                    FACE_SHARPNESS_WEIGHT
                    *
                    face[
                        "sharpness_score"
                    ]
                    +
                    FACE_FRONTAL_WEIGHT
                    *
                    face[
                        "frontal_score"
                    ]
                    +
                    FACE_DETECTION_WEIGHT
                    *
                    face[
                        "det_score"
                    ]
                    +
                    FACE_SIZE_WEIGHT
                    *
                    face[
                        "face_size_score"
                    ]
                    +
                    FACE_EXPOSURE_WEIGHT
                    *
                    face[
                        "exposure_score"
                    ]
                )


        # ----------------------------------------------------
        # FRAME SCORES
        # ----------------------------------------------------

        frame_df = pd.DataFrame(
            [
                {
                    "frame_id":
                        record[
                            "frame_id"
                        ],

                    "source_frame_index":
                        record[
                            "source_frame_index"
                        ],

                    "full_frame_sharpness":
                        record[
                            "full_frame_sharpness"
                        ],

                    "usable_faces":
                        len(
                            record[
                                "faces"
                            ]
                        ),

                    "median_face_quality":
                        (
                            float(
                                np.median(
                                    [
                                        f[
                                            "face_quality_score"
                                        ]
                                        for f
                                        in record[
                                            "faces"
                                        ]
                                    ]
                                )
                            )
                            if record[
                                "faces"
                            ]
                            else np.nan
                        ),

                    "worst_face_quality":
                        (
                            float(
                                np.min(
                                    [
                                        f[
                                            "face_quality_score"
                                        ]
                                        for f
                                        in record[
                                            "faces"
                                        ]
                                    ]
                                )
                            )
                            if record[
                                "faces"
                            ]
                            else np.nan
                        ),

                    "median_frontal_score":
                        (
                            float(
                                np.median(
                                    [
                                        f[
                                            "frontal_score"
                                        ]
                                        for f
                                        in record[
                                            "faces"
                                        ]
                                    ]
                                )
                            )
                            if record[
                                "faces"
                            ]
                            else np.nan
                        ),
                }
                for record
                in frame_records
            ]
        )


        frame_df[
            "full_frame_sharpness_score"
        ] = percentile_rank(
            frame_df[
                "full_frame_sharpness"
            ]
        )

        max_faces = max(
            1,
            int(
                frame_df[
                    "usable_faces"
                ].max()
            ),
        )

        frame_df[
            "face_count_score"
        ] = (
            frame_df[
                "usable_faces"
            ]
            /
            max_faces
        )


        frame_df[
            "frame_quality_score"
        ] = (
            FRAME_MEDIAN_FACE_WEIGHT
            *
            frame_df[
                "median_face_quality"
            ].fillna(
                0
            )
            +
            FRAME_WORST_FACE_WEIGHT
            *
            frame_df[
                "worst_face_quality"
            ].fillna(
                0
            )
            +
            FRAME_FACE_COUNT_WEIGHT
            *
            frame_df[
                "face_count_score"
            ]
            +
            FRAME_SHARPNESS_WEIGHT
            *
            frame_df[
                "full_frame_sharpness_score"
            ]
        )


        frame_df.loc[
            frame_df[
                "usable_faces"
            ]
            ==
            0,
            "frame_quality_score",
        ] = -1.0


        candidate_df = (
            frame_df[
                frame_df[
                    "usable_faces"
                ]
                >
                0
            ]
            .sort_values(
                [
                    "frame_quality_score",
                    "median_face_quality",
                    "full_frame_sharpness",
                ],
                ascending=False,
            )
        )


        if candidate_df.empty:

            pending_window_rows.append({
                "window_id":
                    window_id,

                "range_id":
                    range_id,

                "video_id":
                    video_id,

                "source_video":
                    source_video,

                "source_path":
                    str(
                        source_path
                    ),

                "window_start_frame":
                    int(
                        window_start
                    ),

                "window_end_frame":
                    int(
                        window_end
                    ),

                "best_frame_number":
                    np.nan,

                "second_frame_number":
                    np.nan,

                "best_frame_id":
                    None,

                "best_full_frame_path":
                    None,

                "usable_frames":
                    0,
            })


        else:

            selected = candidate_df.head(
                IDENTITY_TOP_N
            )

            selected_frame_ids = set(
                selected[
                    "frame_id"
                ].astype(
                    str
                )
            )

            best_row = selected.iloc[
                0
            ]

            best_frame_id = str(
                best_row[
                    "frame_id"
                ]
            )

            best_frame_number = int(
                best_row[
                    "source_frame_index"
                ]
            )

            second_frame_number = (
                int(
                    selected.iloc[
                        1
                    ][
                        "source_frame_index"
                    ]
                )
                if len(
                    selected
                ) > 1
                else np.nan
            )


            best_frame_path = (
                FRAMES_DIR /
                f"{best_frame_id}.jpg"
            )


            # Save the best full frame once.
            for record in frame_records:

                if (
                    record[
                        "frame_id"
                    ]
                    ==
                    best_frame_id
                ):

                    cv2.imwrite(
                        str(
                            best_frame_path
                        ),
                        record[
                            "frame"
                        ],
                        [
                            cv2.IMWRITE_JPEG_QUALITY,
                            FULL_FRAME_JPEG_QUALITY,
                        ],
                    )

                    break


            # Save face material only for Top 2 identity frames.
            for record in frame_records:

                if (
                    record[
                        "frame_id"
                    ]
                    not in
                    selected_frame_ids
                ):
                    continue

                identity_rank = int(
                    np.where(
                        selected[
                            "frame_id"
                        ].astype(
                            str
                        ).to_numpy()
                        ==
                        str(
                            record[
                                "frame_id"
                            ]
                        )
                    )[
                        0
                    ][
                        0
                    ]
                    +
                    1
                )


                for face in record[
                    "faces"
                ]:

                    observation_id = str(
                        face[
                            "observation_id"
                        ]
                    )

                    face_crop_path = (
                        FACE_CROPS_DIR /
                        f"{observation_id}.jpg"
                    )

                    aligned_path = (
                        ALIGNED_DIR /
                        f"{observation_id}.jpg"
                    )

                    cv2.imwrite(
                        str(
                            face_crop_path
                        ),
                        face[
                            "_crop"
                        ],
                        [
                            cv2.IMWRITE_JPEG_QUALITY,
                            FACE_CROP_JPEG_QUALITY,
                        ],
                    )

                    cv2.imwrite(
                        str(
                            aligned_path
                        ),
                        face[
                            "_aligned"
                        ],
                        [
                            cv2.IMWRITE_JPEG_QUALITY,
                            ALIGNED_JPEG_QUALITY,
                        ],
                    )


                    pending_face_rows.append({
                        "observation_id":
                            observation_id,

                        "frame_id":
                            face[
                                "frame_id"
                            ],

                        "window_id":
                            window_id,

                        "range_id":
                            range_id,

                        "video_id":
                            video_id,

                        "source_video":
                            source_video,

                        "source_path":
                            str(
                                source_path
                            ),

                        "source_frame_index":
                            int(
                                face[
                                    "source_frame_index"
                                ]
                            ),

                        "source_time_seconds":
                            float(
                                face[
                                    "source_time_seconds"
                                ]
                            ),

                        "reported_fps":
                            fps,

                        "frame_width":
                            int(
                                face[
                                    "frame_width"
                                ]
                            ),

                        "frame_height":
                            int(
                                face[
                                    "frame_height"
                                ]
                            ),

                        "face_bbox_x1":
                            int(
                                face[
                                    "face_bbox_x1"
                                ]
                            ),

                        "face_bbox_y1":
                            int(
                                face[
                                    "face_bbox_y1"
                                ]
                            ),

                        "face_bbox_x2":
                            int(
                                face[
                                    "face_bbox_x2"
                                ]
                            ),

                        "face_bbox_y2":
                            int(
                                face[
                                    "face_bbox_y2"
                                ]
                            ),

                        "center_x_norm":
                            float(
                                face[
                                    "center_x_norm"
                                ]
                            ),

                        "center_y_norm":
                            float(
                                face[
                                    "center_y_norm"
                                ]
                            ),

                        "det_score":
                            float(
                                face[
                                    "det_score"
                                ]
                            ),

                        "sharpness_laplacian":
                            float(
                                face[
                                    "sharpness_laplacian"
                                ]
                            ),

                        "sharpness_score":
                            float(
                                face[
                                    "sharpness_score"
                                ]
                            ),

                        "frontal_score":
                            float(
                                face[
                                    "frontal_score"
                                ]
                            ),

                        "exposure_score":
                            float(
                                face[
                                    "exposure_score"
                                ]
                            ),

                        "face_size_score":
                            float(
                                face[
                                    "face_size_score"
                                ]
                            ),

                        "face_quality_score":
                            float(
                                face[
                                    "face_quality_score"
                                ]
                            ),

                        "identity_frame_rank":
                            identity_rank,

                        "selected_downstream_frame":
                            (
                                str(
                                    face[
                                        "frame_id"
                                    ]
                                )
                                ==
                                best_frame_id
                            ),

                        "full_frame_path":
                            (
                                str(
                                    best_frame_path
                                )
                                if (
                                    str(
                                        face[
                                            "frame_id"
                                        ]
                                    )
                                    ==
                                    best_frame_id
                                )
                                else None
                            ),

                        "face_crop_path":
                            str(
                                face_crop_path
                            ),

                        "identity_aligned_path":
                            str(
                                aligned_path
                            ),
                    })


            pending_window_rows.append({
                "window_id":
                    window_id,

                "range_id":
                    range_id,

                "video_id":
                    video_id,

                "source_video":
                    source_video,

                "source_path":
                    str(
                        source_path
                    ),

                "window_start_frame":
                    int(
                        window_start
                    ),

                "window_end_frame":
                    int(
                        window_end
                    ),

                "best_frame_number":
                    best_frame_number,

                "second_frame_number":
                    second_frame_number,

                "best_frame_id":
                    best_frame_id,

                "best_full_frame_path":
                    str(
                        best_frame_path
                    ),

                "usable_frames":
                    int(
                        len(
                            candidate_df
                        )
                    ),

                "best_frame_quality_score":
                    float(
                        best_row[
                            "frame_quality_score"
                        ]
                    ),

                "best_median_face_quality":
                    float(
                        best_row[
                            "median_face_quality"
                        ]
                    ),

                "best_median_frontal_score":
                    float(
                        best_row[
                            "median_frontal_score"
                        ]
                    ),

                "best_full_frame_sharpness":
                    float(
                        best_row[
                            "full_frame_sharpness"
                        ]
                    ),
            })


        processed_windows.add(
            window_id
        )

        windows_since_checkpoint += 1


        if (
            windows_since_checkpoint
            >=
            CHECKPOINT_WINDOWS
        ):

            append_checkpoint(
                pending_window_rows,
                pending_face_rows,
            )

            pending_window_rows = []
            pending_face_rows = []
            windows_since_checkpoint = 0

            PROCESSED_WINDOWS_JSON.write_text(
                json.dumps(
                    sorted(
                        processed_windows
                    ),
                    indent=2,
                ),
                encoding="utf-8",
            )


    cap.release()


append_checkpoint(
    pending_window_rows,
    pending_face_rows,
)


PROCESSED_WINDOWS_JSON.write_text(
    json.dumps(
        sorted(
            processed_windows
        ),
        indent=2,
    ),
    encoding="utf-8",
)


df_window_results = pd.read_csv(
    WINDOWS_CSV
)

df_identity_faces_raw = pd.read_csv(
    IDENTITY_FACES_CSV
)


print()
print(
    "Processed windows:",
    len(
        df_window_results
    ),
)

print(
    "Top-2 face observations:",
    len(
        df_identity_faces_raw
    ),
)

print(
    "Best downstream frames:",
    int(
        df_window_results[
            "best_frame_id"
        ].notna().sum()
    ),
)

## 6. Load AdaFace IR101 WebFace12M

AdaFace runs **only on faces from the Top 2 frames**, which is the main identity-compute reduction identified in notebook 05.

In [ ]:
ADAFACE_REPO = (
    MODELS /
    "AdaFace"
)

ADAFACE_CHECKPOINT = (
    ADAFACE_REPO /
    "pretrained" /
    "adaface_ir101_webface12m.ckpt"
)


if not ADAFACE_REPO.exists():
    raise FileNotFoundError(
        f"Missing AdaFace repository: {ADAFACE_REPO.resolve()}"
    )


if not ADAFACE_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing AdaFace checkpoint: {ADAFACE_CHECKPOINT.resolve()}"
    )


if str(
    ADAFACE_REPO.resolve()
) not in sys.path:

    sys.path.insert(
        0,
        str(
            ADAFACE_REPO.resolve()
        ),
    )


import net as adaface_net


adaface_model = adaface_net.build_model(
    "ir_101"
)


checkpoint = torch.load(
    ADAFACE_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


state_dict = checkpoint[
    "state_dict"
]


model_state_dict = {
    key[
        6:
    ]:
        value

    for key, value
    in state_dict.items()

    if key.startswith(
        "model."
    )
}


adaface_model.load_state_dict(
    model_state_dict,
    strict=True,
)


def safe_flatten_forward(
    self,
    x,
):
    return x.reshape(
        x.size(
            0
        ),
        -1,
    )


for module in (
    adaface_model.modules()
):

    if (
        module.__class__.__name__
        ==
        "Flatten"
    ):

        module.forward = types.MethodType(
            safe_flatten_forward,
            module,
        )


DEVICE = (
    torch.device(
        "mps"
    )
    if torch.backends.mps.is_available()
    else torch.device(
        "cpu"
    )
)


adaface_model = (
    adaface_model
    .to(
        DEVICE
    )
    .eval()
)


print(
    "AdaFace ready."
)

print(
    "Device:",
    DEVICE,
)

## 7. Generate Top-2 AdaFace embeddings

The embedding matrix is cached. Rerunning this notebook after embeddings exist does not recompute them unless the output directory is reset.

In [ ]:
EMBEDDINGS_NPY = (
    EMBEDDINGS_DIR /
    "top2_adaface_embeddings.npy"
)

EMBED_INDEX_CSV = (
    RESULTS_DIR /
    "top2_embedding_index.csv"
)


def load_aligned_tensor(
    image_path,
):
    image = cv2.imread(
        str(
            image_path
        )
    )

    if image is None:
        raise FileNotFoundError(
            image_path
        )

    image = cv2.resize(
        image,
        (
            112,
            112,
        ),
        interpolation=cv2.INTER_AREA,
    )

    image = (
        image.astype(
            np.float32
        )
        /
        255.0
    )

    image = (
        image
        -
        0.5
    ) / 0.5

    image = image.transpose(
        2,
        0,
        1,
    )

    return np.ascontiguousarray(
        image
    )


def run_adaface(
    tensor,
):
    global DEVICE
    global adaface_model

    try:

        with torch.inference_mode():

            features, _ = (
                adaface_model(
                    tensor
                )
            )

    except RuntimeError:

        if DEVICE.type != "mps":
            raise

        print(
            "MPS failed; switching AdaFace to CPU."
        )

        DEVICE = torch.device(
            "cpu"
        )

        adaface_model = (
            adaface_model
            .to(
                DEVICE
            )
            .eval()
        )

        tensor = tensor.to(
            DEVICE
        )

        with torch.inference_mode():

            features, _ = (
                adaface_model(
                    tensor
                )
            )


    return (
        torch.nn.functional.normalize(
            features,
            p=2,
            dim=1,
        )
    )


if (
    EMBEDDINGS_NPY.exists()
    and
    EMBED_INDEX_CSV.exists()
):

    X_faces = np.load(
        EMBEDDINGS_NPY
    )

    df_embedding_index = pd.read_csv(
        EMBED_INDEX_CSV
    )


else:

    ordered = (
        df_identity_faces_raw
        .sort_values(
            "observation_id"
        )
        .reset_index(
            drop=True
        )
    )

    feature_batches = []
    embedding_ids = []


    for start in tqdm(
        range(
            0,
            len(
                ordered
            ),
            EMBED_BATCH_SIZE,
        ),
        desc="AdaFace Top-2 embeddings",
    ):

        batch_df = ordered.iloc[
            start:
            start
            +
            EMBED_BATCH_SIZE
        ]

        images = []
        ids = []


        for _, row in (
            batch_df.iterrows()
        ):

            try:

                images.append(
                    load_aligned_tensor(
                        row[
                            "identity_aligned_path"
                        ]
                    )
                )

                ids.append(
                    str(
                        row[
                            "observation_id"
                        ]
                    )
                )

            except Exception as exc:

                print(
                    "Skipping:",
                    row[
                        "observation_id"
                    ],
                    exc,
                )


        if not images:
            continue


        tensor = (
            torch.from_numpy(
                np.stack(
                    images
                )
            )
            .float()
            .contiguous()
            .to(
                DEVICE
            )
        )


        features = run_adaface(
            tensor
        )


        feature_batches.append(
            features
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        embedding_ids.extend(
            ids
        )


    if not feature_batches:
        raise RuntimeError(
            "No AdaFace embeddings were generated."
        )


    X_faces = np.vstack(
        feature_batches
    ).astype(
        np.float32
    )


    df_embedding_index = pd.DataFrame({
        "observation_id":
            embedding_ids,

        "embedding_index":
            np.arange(
                len(
                    embedding_ids
                )
            ),
    })


    np.save(
        EMBEDDINGS_NPY,
        X_faces,
    )


    df_embedding_index.to_csv(
        EMBED_INDEX_CSV,
        index=False,
    )


df_faces_embed = (
    df_embedding_index
    .merge(
        df_identity_faces_raw,
        on="observation_id",
        how="left",
        validate="one_to_one",
    )
)


print(
    "Embedding matrix:",
    X_faces.shape,
)

print(
    "Mean norm:",
    float(
        np.linalg.norm(
            X_faces,
            axis=1,
        ).mean()
    ),
)

## 8. Build two-frame temporal tracklets

Within each 15-frame window, faces from the Top 2 selected frames are linked using:

- AdaFace cosine similarity
- spatial continuity
- one-to-one Hungarian matching

Tracklet embeddings are quality-weighted.

In [ ]:
def build_window_tracklets(
    window_df,
):
    tracks = {}
    next_track = 1


    for frame_number, frame_df in (
        window_df
        .sort_values(
            [
                "source_frame_index",
                "observation_id",
            ]
        )
        .groupby(
            "source_frame_index",
            sort=True,
        )
    ):

        detection_rows = list(
            frame_df.index
        )


        active_ids = [
            track_id

            for track_id, state
            in tracks.items()

            if (
                int(
                    frame_number
                )
                -
                state[
                    "last_frame"
                ]
            )
            <=
            MAX_TRACK_GAP_FRAMES
        ]


        matched = set()


        if (
            active_ids
            and
            detection_rows
        ):

            active_centroids = np.stack(
                [
                    tracks[
                        track_id
                    ][
                        "centroid"
                    ]

                    for track_id
                    in active_ids
                ]
            )


            detection_embeddings = X_faces[
                window_df.loc[
                    detection_rows,
                    "embedding_index",
                ].to_numpy(
                    dtype=int
                )
            ]


            similarities = (
                active_centroids
                @
                detection_embeddings.T
            )


            costs = np.full(
                similarities.shape,
                1e6,
                dtype=float,
            )


            for track_pos, track_id in enumerate(
                active_ids
            ):

                state = tracks[
                    track_id
                ]


                for det_pos, row_idx in enumerate(
                    detection_rows
                ):

                    sim = float(
                        similarities[
                            track_pos,
                            det_pos,
                        ]
                    )

                    if (
                        sim
                        <
                        LOCAL_TRACK_SIMILARITY
                    ):
                        continue


                    dx = (
                        float(
                            window_df.loc[
                                row_idx,
                                "center_x_norm",
                            ]
                        )
                        -
                        state[
                            "last_center_x"
                        ]
                    )

                    dy = (
                        float(
                            window_df.loc[
                                row_idx,
                                "center_y_norm",
                            ]
                        )
                        -
                        state[
                            "last_center_y"
                        ]
                    )


                    distance = math.sqrt(
                        dx * dx
                        +
                        dy * dy
                    )


                    if (
                        distance
                        >
                        MAX_CENTER_DISTANCE_NORM
                    ):
                        continue


                    score = (
                        sim
                        -
                        SPATIAL_PENALTY
                        *
                        distance
                    )


                    costs[
                        track_pos,
                        det_pos,
                    ] = (
                        -score
                    )


            row_ind, col_ind = (
                linear_sum_assignment(
                    costs
                )
            )


            for track_pos, det_pos in zip(
                row_ind,
                col_ind,
            ):

                if (
                    costs[
                        track_pos,
                        det_pos,
                    ]
                    >=
                    1e5
                ):
                    continue


                track_id = active_ids[
                    track_pos
                ]

                row_idx = detection_rows[
                    det_pos
                ]

                matched.add(
                    row_idx
                )


                row = window_df.loc[
                    row_idx
                ]

                embedding = X_faces[
                    int(
                        row[
                            "embedding_index"
                        ]
                    )
                ]


                weight = (
                    0.10
                    +
                    float(
                        row[
                            "face_quality_score"
                        ]
                    )
                )


                state = tracks[
                    track_id
                ]

                state[
                    "weighted_sum"
                ] += (
                    embedding
                    *
                    weight
                )

                state[
                    "weight_sum"
                ] += weight

                state[
                    "centroid"
                ] = normalize_vector(
                    state[
                        "weighted_sum"
                    ]
                )

                state[
                    "last_frame"
                ] = int(
                    frame_number
                )

                state[
                    "last_center_x"
                ] = float(
                    row[
                        "center_x_norm"
                    ]
                )

                state[
                    "last_center_y"
                ] = float(
                    row[
                        "center_y_norm"
                    ]
                )

                state[
                    "row_indices"
                ].append(
                    row_idx
                )

                state[
                    "frame_keys"
                ].add(
                    (
                        str(
                            row[
                                "source_video"
                            ]
                        ),
                        int(
                            row[
                                "source_frame_index"
                            ]
                        ),
                    )
                )


        for row_idx in detection_rows:

            if row_idx in matched:
                continue


            row = window_df.loc[
                row_idx
            ]

            embedding = X_faces[
                int(
                    row[
                        "embedding_index"
                    ]
                )
            ]


            weight = (
                0.10
                +
                float(
                    row[
                        "face_quality_score"
                    ]
                )
            )


            track_id = (
                f"T{next_track:03d}"
            )

            next_track += 1


            tracks[
                track_id
            ] = {
                "centroid":
                    normalize_vector(
                        embedding
                    ),

                "weighted_sum":
                    embedding
                    *
                    weight,

                "weight_sum":
                    weight,

                "last_frame":
                    int(
                        frame_number
                    ),

                "last_center_x":
                    float(
                        row[
                            "center_x_norm"
                        ]
                    ),

                "last_center_y":
                    float(
                        row[
                            "center_y_norm"
                        ]
                    ),

                "row_indices":
                    [
                        row_idx
                    ],

                "frame_keys":
                    {
                        (
                            str(
                                row[
                                    "source_video"
                                ]
                            ),
                            int(
                                row[
                                    "source_frame_index"
                                ]
                            ),
                        )
                    },
            }


    return tracks


tracklet_rows = []
membership_rows = []
tracklet_embeddings = []

global_tracklet_number = 1


for window_id, window_df in tqdm(
    df_faces_embed.groupby(
        "window_id"
    ),
    desc="Top-2 temporal tracklets",
):

    window_df = window_df.copy()

    states = build_window_tracklets(
        window_df
    )


    for _, state in (
        states.items()
    ):

        group = (
            window_df.loc[
                state[
                    "row_indices"
                ]
            ]
            .copy()
        )


        pool = (
            group
            .sort_values(
                [
                    "face_quality_score",
                    "sharpness_laplacian",
                    "frontal_score",
                ],
                ascending=False,
            )
            .head(
                TRACKLET_TOP_K
            )
        )


        embed_indices = (
            pool[
                "embedding_index"
            ]
            .to_numpy(
                dtype=int
            )
        )


        weights = (
            0.10
            +
            pool[
                "face_quality_score"
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        pooled = (
            X_faces[
                embed_indices
            ]
            *
            weights[
                :,
                None,
            ]
        ).sum(
            axis=0
        )


        pooled = normalize_vector(
            pooled
        )


        tracklet_id = (
            f"Tracklet_"
            f"{global_tracklet_number:07d}"
        )

        global_tracklet_number += 1


        tracklet_embedding_index = len(
            tracklet_embeddings
        )

        tracklet_embeddings.append(
            pooled.astype(
                np.float32
            )
        )


        best = (
            group
            .sort_values(
                "face_quality_score",
                ascending=False,
            )
            .iloc[
                0
            ]
        )


        tracklet_rows.append({
            "tracklet_id":
                tracklet_id,

            "tracklet_embedding_index":
                tracklet_embedding_index,

            "window_id":
                window_id,

            "source_video":
                best[
                    "source_video"
                ],

            "observations":
                int(
                    len(
                        group
                    )
                ),

            "pool_observations":
                int(
                    len(
                        pool
                    )
                ),

            "best_face_quality":
                float(
                    best[
                        "face_quality_score"
                    ]
                ),

            "best_observation_id":
                str(
                    best[
                        "observation_id"
                    ]
                ),

            "frame_keys_json":
                json.dumps(
                    [
                        [
                            str(v),
                            int(f),
                        ]

                        for v, f
                        in sorted(
                            state[
                                "frame_keys"
                            ]
                        )
                    ]
                ),
        })


        for observation_id in (
            group[
                "observation_id"
            ]
        ):

            membership_rows.append({
                "observation_id":
                    str(
                        observation_id
                    ),

                "tracklet_id":
                    tracklet_id,
            })


df_tracklets = pd.DataFrame(
    tracklet_rows
)

df_track_membership = pd.DataFrame(
    membership_rows
)

X_tracklets = np.vstack(
    tracklet_embeddings
).astype(
    np.float32
)


TRACKLET_CSV = (
    RESULTS_DIR /
    "top2_tracklets.csv"
)

TRACKLET_MEMBERSHIP_CSV = (
    RESULTS_DIR /
    "top2_tracklet_membership.csv"
)

TRACKLET_EMBEDDINGS_NPY = (
    EMBEDDINGS_DIR /
    "top2_tracklet_embeddings.npy"
)


df_tracklets.to_csv(
    TRACKLET_CSV,
    index=False,
)

df_track_membership.to_csv(
    TRACKLET_MEMBERSHIP_CSV,
    index=False,
)

np.save(
    TRACKLET_EMBEDDINGS_NPY,
    X_tracklets,
)


print(
    "Tracklets:",
    len(
        df_tracklets
    ),
)

print(
    "Median observations/tracklet:",
    float(
        df_tracklets[
            "observations"
        ].median()
    ),
)

## 9. Global anonymous Person_ID clustering

Tracklets are clustered across all source videos.

Safety constraint:

> Two different detections from the exact same source frame cannot be assigned to the same Person_ID.

In [ ]:
def parse_frame_keys(
    value,
):
    return {
        (
            str(
                video
            ),
            int(
                frame
            ),
        )

        for video, frame
        in json.loads(
            value
        )
    }


labels = np.full(
    len(
        df_tracklets
    ),
    -1,
    dtype=int,
)

centroids = []
weights = []
frame_sets = []


order = (
    df_tracklets
    .sort_values(
        [
            "observations",
            "best_face_quality",
        ],
        ascending=False,
    )
    .index
    .tolist()
)


for row_index in tqdm(
    order,
    desc="Global Person_ID clustering",
):

    row = df_tracklets.loc[
        row_index
    ]


    embedding = normalize_vector(
        X_tracklets[
            int(
                row[
                    "tracklet_embedding_index"
                ]
            )
        ]
    )


    frame_keys = parse_frame_keys(
        row[
            "frame_keys_json"
        ]
    )


    selected_cluster = None


    if centroids:

        centroid_matrix = np.stack(
            centroids
        )

        similarities = (
            centroid_matrix
            @
            embedding
        )

        ranked = np.argsort(
            -similarities
        )


        for candidate in ranked:

            similarity = float(
                similarities[
                    candidate
                ]
            )

            if (
                similarity
                <
                GLOBAL_TRACKLET_SIMILARITY
            ):
                break


            if (
                frame_sets[
                    candidate
                ]
                &
                frame_keys
            ):
                continue


            selected_cluster = int(
                candidate
            )

            break


    tracklet_weight = max(
        1.0,
        float(
            row[
                "pool_observations"
            ]
        ),
    )


    if selected_cluster is None:

        selected_cluster = len(
            centroids
        )

        centroids.append(
            embedding.copy()
        )

        weights.append(
            tracklet_weight
        )

        frame_sets.append(
            set(
                frame_keys
            )
        )


    else:

        updated = (
            centroids[
                selected_cluster
            ]
            *
            weights[
                selected_cluster
            ]
            +
            embedding
            *
            tracklet_weight
        )

        centroids[
            selected_cluster
        ] = normalize_vector(
            updated
        )

        weights[
            selected_cluster
        ] += (
            tracklet_weight
        )

        frame_sets[
            selected_cluster
        ].update(
            frame_keys
        )


    labels[
        row_index
    ] = (
        selected_cluster
    )


df_tracklets_final = (
    df_tracklets.copy()
)

df_tracklets_final[
    "cluster_label"
] = labels


cluster_ranking = (
    df_tracklets_final
    .groupby(
        "cluster_label"
    )
    .agg(
        observations=(
            "observations",
            "sum",
        ),

        tracklets=(
            "tracklet_id",
            "count",
        ),
    )
    .sort_values(
        [
            "observations",
            "tracklets",
        ],
        ascending=False,
    )
    .reset_index()
)


person_map = {
    int(
        cluster
    ):
        f"Person_{i:04d}"

    for i, cluster
    in enumerate(
        cluster_ranking[
            "cluster_label"
        ],
        start=1,
    )
}


df_tracklets_final[
    "person_id"
] = (
    df_tracklets_final[
        "cluster_label"
    ]
    .map(
        person_map
    )
)


df_final_faces = (
    df_faces_embed
    .merge(
        df_track_membership,
        on="observation_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_tracklets_final[
            [
                "tracklet_id",
                "cluster_label",
                "person_id",
            ]
        ],
        on="tracklet_id",
        how="left",
        validate="many_to_one",
    )
)


same_frame_conflicts = int(
    df_final_faces
    .duplicated(
        [
            "source_video",
            "source_frame_index",
            "person_id",
        ]
    )
    .sum()
)


assert (
    same_frame_conflicts
    ==
    0
), (
    "Same Person_ID assigned to multiple faces "
    "in one source frame."
)


FINAL_FACE_OBSERVATIONS_CSV = (
    RESULTS_DIR /
    "final_identity_observations.csv"
)

df_final_faces.to_csv(
    FINAL_FACE_OBSERVATIONS_CSV,
    index=False,
)


person_summary = (
    df_final_faces
    .groupby(
        "person_id"
    )
    .agg(
        occurrences=(
            "frame_id",
            "nunique",
        ),

        observations=(
            "observation_id",
            "count",
        ),

        videos=(
            "source_video",
            "nunique",
        ),

        mean_face_quality=(
            "face_quality_score",
            "mean",
        ),

        mean_frontal_score=(
            "frontal_score",
            "mean",
        ),

        mean_det_score=(
            "det_score",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "occurrences",
            "videos",
        ],
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


FINAL_SUMMARY_CSV = (
    RESULTS_DIR /
    "final_identity_summary.csv"
)

person_summary.to_csv(
    FINAL_SUMMARY_CSV,
    index=False,
)


singleton_ids = int(
    (
        person_summary[
            "occurrences"
        ]
        ==
        1
    ).sum()
)


print(
    "Final Person_IDs:",
    person_summary[
        "person_id"
    ].nunique(),
)

print(
    "Singleton Person_IDs:",
    singleton_ids,
)

print(
    "Singleton rate:",
    f"{singleton_ids / max(1, len(person_summary)):.1%}",
)

print(
    "Same-frame conflicts:",
    same_frame_conflicts,
)

## 10. Save representative faces

One high-quality face is saved per Person_ID for quick manual inspection.

In [ ]:
for person_id, group in tqdm(
    df_final_faces.groupby(
        "person_id"
    ),
    desc="Representative faces",
):

    best = (
        group
        .sort_values(
            [
                "face_quality_score",
                "sharpness_laplacian",
                "frontal_score",
            ],
            ascending=False,
        )
        .iloc[
            0
        ]
    )


    source = Path(
        str(
            best[
                "face_crop_path"
            ]
        )
    )


    destination = (
        REP_FACES_DIR /
        f"{person_id}.jpg"
    )


    if source.exists():
        shutil.copy2(
            source,
            destination,
        )


print(
    "Representative faces:",
    REP_FACES_DIR.resolve(),
)

# Team handoffs

The following cells create the files that Marion and RJ should use.

Only the **best frame of each 15-frame window** is included in downstream handoffs.

PersonID itself was determined using the **Top 2** frames.

## 11. RJ — `sentiment_input_clean.csv`

Schema matches the supplied RJ file exactly:

```text
video_title
video_filename
frame_number
person_id
image_path
source_time_seconds
det_score
```

`image_path` begins with `FaceCrops/`.

In [ ]:
downstream_faces = (
    df_final_faces[
        df_final_faces[
            "selected_downstream_frame"
        ].astype(
            bool
        )
    ]
    .copy()
)


# Defensive rule: one Person_ID at most once per frame.
downstream_faces = (
    downstream_faces
    .sort_values(
        [
            "face_quality_score",
            "det_score",
        ],
        ascending=False,
    )
    .drop_duplicates(
        [
            "source_video",
            "source_frame_index",
            "person_id",
        ],
        keep="first",
    )
)


downstream_faces[
    "video_title"
] = (
    downstream_faces[
        "source_video"
    ]
    .apply(
        lambda x:
            Path(
                str(
                    x
                )
            ).stem
    )
)


downstream_faces[
    "video_filename"
] = (
    downstream_faces[
        "source_video"
    ].astype(
        str
    )
)


downstream_faces[
    "frame_number"
] = (
    downstream_faces[
        "source_frame_index"
    ].astype(
        int
    )
)


downstream_faces[
    "image_path"
] = (
    downstream_faces[
        "face_crop_path"
    ]
    .apply(
        lambda x:
            "FaceCrops/"
            +
            Path(
                str(
                    x
                )
            ).name
    )
)


sentiment_input_clean = (
    downstream_faces[
        [
            "video_title",
            "video_filename",
            "frame_number",
            "person_id",
            "image_path",
            "source_time_seconds",
            "det_score",
        ]
    ]
    .sort_values(
        [
            "video_title",
            "frame_number",
            "person_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


SENTIMENT_HANDOFF_CSV = (
    HANDOFF_DIR /
    "sentiment_input_clean.csv"
)


sentiment_input_clean.to_csv(
    SENTIMENT_HANDOFF_CSV,
    index=False,
)


expected_rj_columns = [
    "video_title",
    "video_filename",
    "frame_number",
    "person_id",
    "image_path",
    "source_time_seconds",
    "det_score",
]


assert (
    list(
        sentiment_input_clean.columns
    )
    ==
    expected_rj_columns
)


print(
    "RJ rows:",
    len(
        sentiment_input_clean
    ),
)

print(
    "Saved:",
    SENTIMENT_HANDOFF_CSV,
)

display(
    sentiment_input_clean.head(
        10
    )
)

## 12. Marion — `activity_frame_person_map.csv`

Schema matches the supplied Marion map exactly:

```text
video_title
video_filename
frame_number
frame_id
person_id
source_time_seconds
full_frame_path
face_crop_path
face_bbox_x1
face_bbox_y1
face_bbox_x2
face_bbox_y2
det_score
```

In [ ]:
activity_frame_person_map = (
    downstream_faces[
        [
            "video_title",
            "video_filename",
            "frame_number",
            "frame_id",
            "person_id",
            "source_time_seconds",
            "full_frame_path",
            "face_crop_path",
            "face_bbox_x1",
            "face_bbox_y1",
            "face_bbox_x2",
            "face_bbox_y2",
            "det_score",
        ]
    ]
    .sort_values(
        [
            "video_title",
            "frame_number",
            "person_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


ACTIVITY_MAP_CSV = (
    HANDOFF_DIR /
    "activity_frame_person_map.csv"
)


activity_frame_person_map.to_csv(
    ACTIVITY_MAP_CSV,
    index=False,
)


expected_marion_columns = [
    "video_title",
    "video_filename",
    "frame_number",
    "frame_id",
    "person_id",
    "source_time_seconds",
    "full_frame_path",
    "face_crop_path",
    "face_bbox_x1",
    "face_bbox_y1",
    "face_bbox_x2",
    "face_bbox_y2",
    "det_score",
]


assert (
    list(
        activity_frame_person_map.columns
    )
    ==
    expected_marion_columns
)


print(
    "Marion person-frame rows:",
    len(
        activity_frame_person_map
    ),
)

print(
    "Saved:",
    ACTIVITY_MAP_CSV,
)

display(
    activity_frame_person_map.head(
        10
    )
)

## 13. Marion — unique activity frames

Marion should run the activity model on this file, **not once per row of the person map**.

Each source frame appears exactly once.

`persons_json` contains every Person_ID and face bounding box visible in that selected frame.

In [ ]:
activity_rows = []


for (
    video_filename,
    frame_number,
), group in (
    activity_frame_person_map
    .groupby(
        [
            "video_filename",
            "frame_number",
        ],
        sort=False,
    )
):

    first = group.iloc[
        0
    ]

    people = []


    for _, row in (
        group.iterrows()
    ):

        people.append({
            "person_id":
                str(
                    row[
                        "person_id"
                    ]
                ),

            "face_bbox": {
                "x1":
                    int(
                        row[
                            "face_bbox_x1"
                        ]
                    ),

                "y1":
                    int(
                        row[
                            "face_bbox_y1"
                        ]
                    ),

                "x2":
                    int(
                        row[
                            "face_bbox_x2"
                        ]
                    ),

                "y2":
                    int(
                        row[
                            "face_bbox_y2"
                        ]
                    ),
            },
        })


    activity_rows.append({
        "video_title":
            str(
                first[
                    "video_title"
                ]
            ),

        "video_filename":
            str(
                video_filename
            ),

        "frame_number":
            int(
                frame_number
            ),

        "frame_id":
            str(
                first[
                    "frame_id"
                ]
            ),

        "source_time_seconds":
            float(
                first[
                    "source_time_seconds"
                ]
            ),

        "full_frame_path":
            str(
                first[
                    "full_frame_path"
                ]
            ),

        "person_count":
            int(
                len(
                    group
                )
            ),

        "persons_json":
            json.dumps(
                people
            ),
    })


activity_frames_unique = (
    pd.DataFrame(
        activity_rows
    )
    .sort_values(
        [
            "video_title",
            "frame_number",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    activity_frames_unique[
        [
            "video_filename",
            "frame_number",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


ACTIVITY_UNIQUE_CSV = (
    HANDOFF_DIR /
    "activity_frames_unique.csv"
)


activity_frames_unique.to_csv(
    ACTIVITY_UNIQUE_CSV,
    index=False,
)


print(
    "Unique activity model calls:",
    len(
        activity_frames_unique
    ),
)

print(
    "Mapped person appearances:",
    int(
        activity_frames_unique[
            "person_count"
        ].sum()
    ),
)

print(
    "Saved:",
    ACTIVITY_UNIQUE_CSV,
)

display(
    activity_frames_unique.head(
        10
    )
)

## 14. Validate the team handoffs

This checks:

- RJ schema is exact
- Marion schema is exact
- one Person_ID appears at most once per selected frame
- Marion's unique-frame file contains no duplicate frames
- every RJ image path starts with `FaceCrops/`
- all saved paths required by Marion exist

In [ ]:
validation = {}


validation[
    "rj_schema_exact"
] = (
    list(
        sentiment_input_clean.columns
    )
    ==
    expected_rj_columns
)


validation[
    "marion_schema_exact"
] = (
    list(
        activity_frame_person_map.columns
    )
    ==
    expected_marion_columns
)


validation[
    "duplicate_person_per_frame"
] = int(
    activity_frame_person_map
    .duplicated(
        [
            "video_filename",
            "frame_number",
            "person_id",
        ]
    )
    .sum()
)


validation[
    "duplicate_activity_frames"
] = int(
    activity_frames_unique
    .duplicated(
        [
            "video_filename",
            "frame_number",
        ]
    )
    .sum()
)


validation[
    "rj_bad_image_prefix"
] = int(
    (
        ~sentiment_input_clean[
            "image_path"
        ]
        .astype(
            str
        )
        .str.startswith(
            "FaceCrops/"
        )
    ).sum()
)


validation[
    "missing_marion_full_frames"
] = int(
    sum(
        not Path(
            str(
                path
            )
        ).exists()

        for path in activity_frames_unique[
            "full_frame_path"
        ]
    )
)


validation[
    "missing_face_crops"
] = int(
    sum(
        not Path(
            str(
                path
            )
        ).exists()

        for path in activity_frame_person_map[
            "face_crop_path"
        ]
    )
)


assert validation[
    "rj_schema_exact"
]

assert validation[
    "marion_schema_exact"
]

assert (
    validation[
        "duplicate_person_per_frame"
    ]
    ==
    0
)

assert (
    validation[
        "duplicate_activity_frames"
    ]
    ==
    0
)

assert (
    validation[
        "rj_bad_image_prefix"
    ]
    ==
    0
)

assert (
    validation[
        "missing_marion_full_frames"
    ]
    ==
    0
)

assert (
    validation[
        "missing_face_crops"
    ]
    ==
    0
)


print(
    json.dumps(
        validation,
        indent=2,
    )
)

print()
print(
    "TEAM HANDOFF VALIDATION: PASSED"
)

## 15. Final production report

In [ ]:
person_count = int(
    person_summary[
        "person_id"
    ].nunique()
)


report = {
    "run_mode":
        RUN_MODE,

    "frames_per_window":
        FRAMES_PER_WINDOW,

    "identity_top_n":
        IDENTITY_TOP_N,

    "downstream_top_n":
        DOWNSTREAM_TOP_N,

    "global_similarity_threshold":
        GLOBAL_TRACKLET_SIMILARITY,

    "windows_processed":
        int(
            len(
                df_window_results
            )
        ),

    "top2_identity_face_observations":
        int(
            len(
                df_final_faces
            )
        ),

    "tracklets":
        int(
            len(
                df_tracklets_final
            )
        ),

    "person_ids":
        person_count,

    "singleton_person_ids":
        singleton_ids,

    "singleton_rate":
        float(
            singleton_ids
            /
            max(
                1,
                person_count
            )
        ),

    "same_frame_identity_conflicts":
        same_frame_conflicts,

    "rj_sentiment_rows":
        int(
            len(
                sentiment_input_clean
            )
        ),

    "marion_person_frame_rows":
        int(
            len(
                activity_frame_person_map
            )
        ),

    "marion_unique_activity_frames":
        int(
            len(
                activity_frames_unique
            )
        ),

    "handoffs": {
        "rj":
            str(
                SENTIMENT_HANDOFF_CSV
            ),

        "marion_map":
            str(
                ACTIVITY_MAP_CSV
            ),

        "marion_unique_frames":
            str(
                ACTIVITY_UNIQUE_CSV
            ),
    },

    "validation":
        validation,
}


REPORT_JSON = (
    RESULTS_DIR /
    "production_report.json"
)


REPORT_JSON.write_text(
    json.dumps(
        report,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    json.dumps(
        report,
        indent=2,
    )
)


print()
print(
    "=============================================="
)

print(
    "FINAL TEAM FILES"
)

print(
    "=============================================="
)

print(
    "RJ:",
    SENTIMENT_HANDOFF_CSV,
)

print(
    "Marion person map:",
    ACTIVITY_MAP_CSV,
)

print(
    "Marion unique frames:",
    ACTIVITY_UNIQUE_CSV,
)